<div style="background:linear-gradient(125deg,#14283f,#176b72);padding:30px;border-radius:16px;color:#ffffff;">
  <p style="letter-spacing:2px;margin:0;font-size:12px;">DATHEON · MACHINE LEARNING & DATA SCIENCE INTERNSHIP</p>
  <h1 style="margin:10px 0 6px;font-size:30px;">02 · Iris Dataset Cleaning & Quality Assessment</h1>
  <p style="margin:0;font-size:15px;">A structured data-quality workflow using Python and Pandas</p>
  <p style="margin:18px 0 0;font-size:12px;">RAW DATA → PROFILE → VALIDATE → CLEAN → VERIFY → EXPORT</p>
</div>

**Project:** Iris Multi-Class Classification  
**Notebook:** `02_Data_Cleaning_and_Inspection.ipynb`  
**Primary library:** Pandas  
**Input:** `data/raw/iris.data`  
**Output:** `data/processed/iris_cleaned.csv`

# 1. Executive Summary

This notebook prepares the Iris dataset for subsequent exploratory data analysis and classification. It focuses on data quality, consistency, validation, and reproducible cleaning decisions.

The objective is **not to change data just to make it look clean**. Each transformation must have a reason, be documented, and be checked after execution.

### Business context

A hypothetical flower research or nursery team may use flower measurements to study species characteristics and later support species classification. Before analysis, the team needs confidence that measurements are numeric, labels are consistent, and missing or invalid records have been investigated.

*This is an illustrative business use case. The Iris dataset is a small educational benchmark, not a production nursery database.*

### Scope

**Included:** raw data loading, profiling, missing-value analysis, duplicate investigation, type validation, range checks, category validation, descriptive statistics, group-wise summaries, cleaning, final validation, and export.

**Not included:** EDA visualizations, feature scaling, train-test splitting, model training, and model evaluation. Those belong in later notebooks.

# 2. Learning Objectives

By completing this notebook, you should be able to:

1. Load the original raw Iris file without modifying it.
2. inspect structure, data types, missingness, duplicates, and category values.
3. distinguish data errors from legitimate observations.
4. use Pandas aggregation tools such as `groupby()`, `agg()`, `value_counts()`, and `describe()`.
5. apply minimal, justified cleaning transformations.
6. validate the cleaned dataset against explicit quality rules.
7. save a processed dataset and a data-quality report in the Task-01 folder.

# 3. Project Folder and File Locations

The expected structure is:

```text
Task-01-Iris-Classification/
│
├── data/
│   ├── raw/
│   │   └── iris.data
│   └── processed/
│       ├── iris_cleaned.csv
│       ├── data_quality_report.csv
│       └── species_summary.csv
│
├── Jupyter-Notebook-Iris-Project/
│   ├── 01_Data_Loading_and_Inspection.ipynb
│   └── 02_Data_Cleaning_and_Inspection.ipynb
│
└── outputs/
    └── reports/
```

This notebook searches the current working directory and its parent directories for the existing Task-01 data folder. It is designed for the common setup where Jupyter runs from `Jupyter-Notebook-Iris-Project`.

**Important:** Keep `data/raw/iris.data` unchanged. The cleaned output is written separately to `data/processed/`.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

# Locate the Task-01 directory from the notebook's current working directory.
CURRENT_DIR = Path.cwd().resolve()

candidate_dirs = [CURRENT_DIR, *CURRENT_DIR.parents]
TASK_DIR = None

for candidate in candidate_dirs:
    if (candidate / "data").exists():
        TASK_DIR = candidate
        break

# Fallback for a notebook launched from the notebook-project folder.
if TASK_DIR is None and CURRENT_DIR.name.lower() in {
    "jupyter-notebook-iris-project", "notebooks", "notebook"
}:
    TASK_DIR = CURRENT_DIR.parent

if TASK_DIR is None:
    raise FileNotFoundError(
        "Could not locate the Task-01 folder. Run Jupyter from "
        "Task-01-Iris-Classification or its notebook subfolder."
    )

DATA_DIR = TASK_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_FILE = RAW_DIR / "iris.data"
CLEAN_FILE = PROCESSED_DIR / "iris_cleaned.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:", CURRENT_DIR)
print("Task directory:", TASK_DIR)
print("Raw dataset path:", RAW_FILE)
print("Processed output directory:", PROCESSED_DIR)

Current working directory: D:\Datheon-ML-Data-Science-Internship\Task-01-Iris-Classification\Jupyter-Notebook-Iris-Project
Task directory: D:\Datheon-ML-Data-Science-Internship\Task-01-Iris-Classification
Raw dataset path: D:\Datheon-ML-Data-Science-Internship\Task-01-Iris-Classification\data\raw\iris.data
Processed output directory: D:\Datheon-ML-Data-Science-Internship\Task-01-Iris-Classification\data\processed


# 4. Import and Load the Raw Dataset

The UCI Iris source file is a headerless comma-separated file. Therefore, we explicitly provide the five column names while reading it.

The original UCI data contains four measurements in centimetres and one species label. We preserve the raw file and load it into a DataFrame for inspection.

In [2]:
UCI_IRIS_URL = (
    "https://archive.ics.uci.edu/ml/"
    "machine-learning-databases/iris/iris.data"
)

if not RAW_FILE.exists():
    try:
        urlretrieve(UCI_IRIS_URL, RAW_FILE)
        print("Downloaded the original UCI Iris file.")
    except Exception as exc:
        raise RuntimeError(
            f"Raw Iris file was not found at {RAW_FILE}. "
            f"Download it from {UCI_IRIS_URL} and place it in data/raw/."
        ) from exc
else:
    print("Using existing raw dataset:", RAW_FILE)

COLUMN_NAMES = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm",
    "species"
]

raw_df = pd.read_csv(
    RAW_FILE,
    header=None,
    names=COLUMN_NAMES,
    skip_blank_lines=True
)

df = raw_df.copy(deep=True)

print("Raw data loaded.")
print("Shape:", df.shape)
display(df.head())

Using existing raw dataset: D:\Datheon-ML-Data-Science-Internship\Task-01-Iris-Classification\data\raw\iris.data
Raw data loaded.
Shape: (150, 5)


,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
0,5.100,3.500,1.400,0.200,Iris-setosa
1,4.900,3.000,1.400,0.200,Iris-setosa
2,4.700,3.200,1.300,0.200,Iris-setosa
3,4.600,3.100,1.500,0.200,Iris-setosa
4,5.000,3.600,1.400,0.200,Iris-setosa


# 5. Initial Dataset Profile

Before cleaning, record the original dimensions, columns, data types, and memory usage. This creates a baseline for comparing the dataset before and after cleaning.

In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataFrame information:")
df.info()

profile = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "non_null_count": df.notna().sum(),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=True)
})

display(profile)

Rows: 150
Columns: 5

Column names:
['sepal_length_cm', 'sepal_width_cm', 'petal_length_cm', 'petal_width_cm', 'species']

DataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sepal_length_cm  150 non-null    float64
 1   sepal_width_cm   150 non-null    float64
 2   petal_length_cm  150 non-null    float64
 3   petal_width_cm   150 non-null    float64
 4   species          150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 7.9 KB


,data_type,non_null_count,missing_count,missing_percent,unique_count
sepal_length_cm,float64,150,0,0.000,35
sepal_width_cm,float64,150,0,0.000,23
petal_length_cm,float64,150,0,0.000,43
petal_width_cm,float64,150,0,0.000,22
species,str,150,0,0.000,3


# 6. Data Dictionary and Expected Schema

| Column | Meaning | Expected type | Expected rule |
|---|---|---|---|
| `sepal_length_cm` | Sepal length | Numeric | Positive measurement |
| `sepal_width_cm` | Sepal width | Numeric | Positive measurement |
| `petal_length_cm` | Petal length | Numeric | Positive measurement |
| `petal_width_cm` | Petal width | Numeric | Positive measurement |
| `species` | Iris species label | Category / text | One of the three known labels |

**Schema validation** checks whether the expected columns are present and whether the dataset contains unexpected columns. It is better to stop and investigate a schema mismatch than silently continue.

In [4]:
expected_columns = COLUMN_NAMES

missing_columns = sorted(set(expected_columns) - set(df.columns))
unexpected_columns = sorted(set(df.columns) - set(expected_columns))

print("Missing expected columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

if unexpected_columns:
    print("Review unexpected columns before continuing.")
else:
    print("Schema check passed.")

Missing expected columns: []
Unexpected columns: []
Schema check passed.


# 7. Missing-Value Assessment

Missing values may occur because of incomplete data collection, file-format problems, or parsing errors.

We measure missing values by column and by row. We do not automatically delete records or fill values before understanding the extent and likely cause of missingness.

In [5]:
missing_by_column = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

missing_by_row = df.isna().sum(axis=1)

print("Total missing cells:", int(df.isna().sum().sum()))
print("Rows containing at least one missing value:", int((missing_by_row > 0).sum()))

display(missing_by_column)

Total missing cells: 0
Rows containing at least one missing value: 0


,missing_count,missing_percent
sepal_length_cm,0,0.000
sepal_width_cm,0,0.000
petal_length_cm,0,0.000
petal_width_cm,0,0.000
species,0,0.000


## 7.1 Missing-value treatment decision

- If no missing values are found, no imputation is required.
- If numeric measurements are missing, investigate their source before choosing a treatment.
- If the target `species` is missing, the row cannot be used as a labelled example for supervised classification.
- Any imputation strategy should be justified and fitted using training data only when it becomes part of a machine-learning pipeline.

For this small benchmark, this notebook will not invent replacement measurements. It will report missing values and remove rows only if they cannot be used or validated, documenting the count.

# 8. Duplicate-Record Investigation

An exact duplicate is a row in which all column values match another row. However, identical flower measurements do not necessarily prove that a record is erroneous: separate flowers can have the same recorded measurements.

Therefore, this notebook reports duplicates and keeps them by default. Remove them only if the data provider or collection process confirms that they are accidental repeated records.

In [6]:
duplicate_mask = df.duplicated(keep=False)
duplicate_rows = df.loc[duplicate_mask].sort_values(COLUMN_NAMES)

print("Rows participating in exact duplicate groups:", int(duplicate_mask.sum()))
print("Number of duplicate rows after the first occurrence:", int(df.duplicated().sum()))

display(duplicate_rows.head(20))

Rows participating in exact duplicate groups: 5
Number of duplicate rows after the first occurrence: 3


,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
9,4.900,3.100,1.500,0.100,Iris-setosa
34,4.900,3.100,1.500,0.100,Iris-setosa
37,4.900,3.100,1.500,0.100,Iris-setosa
101,5.800,2.700,5.100,1.900,Iris-virginica
142,5.800,2.700,5.100,1.900,Iris-virginica


## 8.1 Duplicate decision

**Default decision: retain exact duplicates unless their accidental duplication is confirmed.**

This is important for Iris because multiple flowers may share identical measurements. Blindly applying `drop_duplicates()` can remove valid observations and alter class counts.

# 9. Data-Type Validation and Conversion

The four measurement fields must be numeric. The species column must be text-like and contain known labels.

`pd.to_numeric(errors="coerce")` converts valid numeric values and turns unparseable values into missing values. We then count those newly introduced missing values so parsing problems are visible.

In [7]:
measurement_columns = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm"
]

before_missing = df[measurement_columns].isna().sum()

for column in measurement_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["species"] = df["species"].astype("string").str.strip()

after_missing = df[measurement_columns].isna().sum()
conversion_issue_report = pd.DataFrame({
    "missing_before_conversion": before_missing,
    "missing_after_conversion": after_missing,
    "additional_missing_after_conversion": after_missing - before_missing
})

display(conversion_issue_report)
display(df.dtypes)

,missing_before_conversion,missing_after_conversion,additional_missing_after_conversion
sepal_length_cm,0,0,0
sepal_width_cm,0,0,0
petal_length_cm,0,0,0
petal_width_cm,0,0,0


sepal_length_cm    float64
sepal_width_cm     float64
petal_length_cm    float64
petal_width_cm     float64
species             string
dtype: object

# 10. Category and Label Validation

The expected species are:

- `Iris-setosa`
- `Iris-versicolor`
- `Iris-virginica`

We inspect observed labels before applying any mapping. Trimming leading/trailing whitespace is a safe formatting normalization; changing a label that does not match the expected set requires investigation.

In [8]:
EXPECTED_SPECIES = {
    "Iris-setosa",
    "Iris-versicolor",
    "Iris-virginica"
}

print("Observed species values:")
display(df["species"].value_counts(dropna=False).to_frame("count"))

observed_species = set(df["species"].dropna().unique())
unexpected_species = sorted(observed_species - EXPECTED_SPECIES)

print("Unexpected species labels:", unexpected_species)
print("Missing species labels:", int(df["species"].isna().sum()))

Observed species values:


,count
species,
Iris-setosa,50
Iris-versicolor,50
Iris-virginica,50


Unexpected species labels: []
Missing species labels: 0


# 11. Numeric Range and Validity Checks

All four Iris measurements are physical lengths in centimetres. A zero or negative length is not a valid physical measurement.

We check:
- non-numeric or missing measurement values;
- zero and negative values;
- unusually small or large values using descriptive statistics.

Do not remove a value merely because it is far from the mean. Statistical unusualness is not automatically a data-entry error.

In [9]:
range_report = pd.DataFrame({
    "minimum": df[measurement_columns].min(),
    "maximum": df[measurement_columns].max(),
    "mean": df[measurement_columns].mean(),
    "median": df[measurement_columns].median(),
    "standard_deviation": df[measurement_columns].std()
})

non_positive_counts = (df[measurement_columns] <= 0).sum()
range_report["non_positive_count"] = non_positive_counts

display(range_report)

invalid_measurement_mask = (df[measurement_columns] <= 0).any(axis=1)
print("Rows containing zero or negative measurements:", int(invalid_measurement_mask.sum()))
display(df.loc[invalid_measurement_mask])

,minimum,maximum,mean,median,standard_deviation,non_positive_count
sepal_length_cm,4.300,7.900,5.843,5.800,0.828,0
sepal_width_cm,2.000,4.400,3.054,3.000,0.434,0
petal_length_cm,1.000,6.900,3.759,4.350,1.764,0
petal_width_cm,0.100,2.500,1.199,1.300,0.763,0


Rows containing zero or negative measurements: 0


,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species


# 12. Descriptive Statistics – Detailed Review

Descriptive statistics help understand the scale and spread of each measurement.

Key measures:

- **Count:** number of non-missing observations.
- **Mean:** arithmetic average.
- **Median:** middle value; less sensitive to extreme values than the mean.
- **Standard deviation:** spread around the mean.
- **Minimum / Maximum:** observed endpoints.
- **Quartiles:** values at 25%, 50%, and 75%.
- **IQR:** difference between the third and first quartiles.

These statistics describe the dataset; they do not by themselves decide which observations should be removed.

In [10]:
descriptive_stats = df[measurement_columns].describe().T
descriptive_stats["median"] = df[measurement_columns].median()
descriptive_stats["variance"] = df[measurement_columns].var()
descriptive_stats["range"] = (
    df[measurement_columns].max() - df[measurement_columns].min()
)
descriptive_stats["IQR"] = (
    df[measurement_columns].quantile(0.75)
    - df[measurement_columns].quantile(0.25)
)

display(descriptive_stats.round(3))

,count,mean,std,min,25%,50%,75%,max,median,variance,range,IQR
sepal_length_cm,150.000,5.843,0.828,4.300,5.100,5.800,6.400,7.900,5.800,0.686,3.600,1.300
sepal_width_cm,150.000,3.054,0.434,2.000,2.800,3.000,3.300,4.400,3.000,0.188,2.400,0.500
petal_length_cm,150.000,3.759,1.764,1.000,1.600,4.350,5.100,6.900,4.350,3.113,5.900,3.500
petal_width_cm,150.000,1.199,0.763,0.100,0.300,1.300,1.800,2.500,1.300,0.582,2.400,1.500


# 13. GroupBy and Aggregation Analysis

`groupby()` splits records into groups, applies calculations to each group, and combines the results.

For this dataset, grouping by `species` helps compare the measurement profile of each flower species.

### Common aggregation functions

| Function | Purpose |
|---|---|
| `count` | Number of non-missing values |
| `mean` | Average |
| `median` | Middle value |
| `min` / `max` | Smallest / largest |
| `std` | Standard deviation |
| `var` | Variance |
| `sum` | Total |
| `nunique` | Number of distinct values |
| `agg()` | Apply multiple aggregations |

`groupby()` and `agg()` are analytical tools here. They help understand the data and are not themselves cleaning operations.

In [11]:
# Number of records in each species
species_counts = df["species"].value_counts(dropna=False)
display(species_counts.to_frame("record_count"))

# Multiple statistics for every measurement, grouped by species
species_aggregation = (
    df.groupby("species", dropna=False)[measurement_columns]
      .agg(["count", "mean", "median", "std", "min", "max"])
)

display(species_aggregation.round(3))

,record_count
species,
Iris-setosa,50
Iris-versicolor,50
Iris-virginica,50


sepal_length_cm                                sepal_width_cm                                petal_length_cm               \
                          count  mean median   std   min   max          count  mean median   std   min   max           count  mean median   
species                                                                                                                                     
Iris-setosa                  50 5.006  5.000 0.352 4.300 5.800             50 3.418  3.400 0.381 2.300 4.400              50 1.464  1.500   
Iris-versicolor              50 5.936  5.900 0.516 4.900 7.000             50 2.770  2.800 0.314 2.000 3.400              50 4.260  4.350   
Iris-virginica               50 6.588  6.500 0.636 4.900 7.900             50 2.974  3.000 0.322 2.200 3.800              50 5.552  5.550   

                                  petal_width_cm                                 
                  std   min   max          count  mean median   std   min   max  
species                                                                          
Iris-setosa     0.174 1.000 1.900             50 0.244  0.200 0.107 0.100 0.600  
Iris-versicolor 0.470 3.000 5.100             50 1.326  1.300 0.198 1.000 1.800  
Iris-virginica  0.552 4.500 6.900             50 2.026  2.000 0.275 1.400 2.500

## 13.1 Named aggregation with `agg()`

Named aggregation creates a clear summary table with meaningful output column names. This format is useful for business reporting and exporting.

In [12]:
species_business_summary = (
    df.groupby("species", dropna=False)
      .agg(
          flower_records=("species", "size"),
          average_sepal_length=("sepal_length_cm", "mean"),
          median_sepal_length=("sepal_length_cm", "median"),
          average_sepal_width=("sepal_width_cm", "mean"),
          average_petal_length=("petal_length_cm", "mean"),
          average_petal_width=("petal_width_cm", "mean"),
          min_petal_length=("petal_length_cm", "min"),
          max_petal_length=("petal_length_cm", "max")
      )
      .reset_index()
)

display(species_business_summary.round(3))

,species,flower_records,average_sepal_length,median_sepal_length,average_sepal_width,average_petal_length,average_petal_width,min_petal_length,max_petal_length
0,Iris-setosa,50,5.006,5.000,3.418,1.464,0.244,1.000,1.900
1,Iris-versicolor,50,5.936,5.900,2.770,4.260,1.326,3.000,5.100
2,Iris-virginica,50,6.588,6.500,2.974,5.552,2.026,4.500,6.900


## 13.2 Grouping by more than one column

Multiple grouping columns can be used when a dataset has more than one category. Iris has only one categorical field in this task, so the example below demonstrates grouping by species and a derived sepal-length band for descriptive analysis.

The band is an analysis label only; it is not a replacement for the original measurement.

In [13]:
df_grouping_demo = df.copy()

df_grouping_demo["sepal_length_band"] = pd.cut(
    df_grouping_demo["sepal_length_cm"],
    bins=[0, 5, 6, 7, np.inf],
    labels=["Up to 5 cm", "5–6 cm", "6–7 cm", "Above 7 cm"],
    right=True
)

grouped_band_summary = (
    df_grouping_demo
    .groupby(["species", "sepal_length_band"], observed=False)
    .size()
    .rename("record_count")
    .reset_index()
)

display(grouped_band_summary.head(20))

,species,sepal_length_band,record_count
0,Iris-setosa,Up to 5 cm,28
1,Iris-setosa,5–6 cm,22
2,Iris-setosa,6–7 cm,0
3,Iris-setosa,Above 7 cm,0
4,Iris-versicolor,Up to 5 cm,3
5,Iris-versicolor,5–6 cm,27
6,Iris-versicolor,6–7 cm,20
7,Iris-versicolor,Above 7 cm,0
8,Iris-virginica,Up to 5 cm,1
9,Iris-virginica,5–6 cm,8


# 14. Outlier Screening Using the IQR Rule

The IQR rule flags values below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR`.

This is an **outlier screening method**, not an automatic deletion rule. A flagged flower measurement may be biologically valid. We will report the number of flags and retain the observations unless there is independent evidence of an error.

In [14]:
outlier_rows = []

for column in measurement_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    flagged = (df[column] < lower_bound) | (df[column] > upper_bound)

    outlier_rows.append({
        "feature": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "flagged_count": int(flagged.sum())
    })

outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report.round(3))

,feature,Q1,Q3,IQR,lower_bound,upper_bound,flagged_count
0,sepal_length_cm,5.100,6.400,1.300,3.150,8.350,0
1,sepal_width_cm,2.800,3.300,0.500,2.050,4.050,4
2,petal_length_cm,1.600,5.100,3.500,-3.650,10.350,0
3,petal_width_cm,0.300,1.800,1.500,-1.950,4.050,0


# 15. Define the Cleaning Rules

Based on the checks above, use the following conservative rules:

1. Keep only the required five columns and preserve their order.
2. Convert the four measurement columns to numeric values.
3. Trim leading/trailing whitespace from species labels.
4. Treat missing measurement values and missing target labels as unusable for this labelled classification dataset; report any rows removed.
5. Remove rows with non-positive physical measurements; report any rows removed.
6. Investigate unknown species labels. Do not silently map them to a known class.
7. Retain exact duplicate records unless accidental duplication is confirmed.
8. Retain IQR-flagged observations unless independent evidence shows an error.

For the standard UCI Iris file, these checks are expected to show a complete dataset. If the results differ, inspect the raw file and document why.

In [15]:
# Work on a copy; never overwrite the raw dataset.
clean_df = df[expected_columns].copy()

rows_before_cleaning = len(clean_df)

# Standardize surrounding whitespace in target labels.
clean_df["species"] = clean_df["species"].astype("string").str.strip()

# Ensure measurement columns are numeric.
for column in measurement_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

# Identify rows that cannot be used for supervised classification.
missing_required_mask = (
    clean_df[measurement_columns].isna().any(axis=1)
    | clean_df["species"].isna()
)

invalid_measurement_mask = (clean_df[measurement_columns] <= 0).any(axis=1)

unknown_species_mask = (
    clean_df["species"].notna()
    & ~clean_df["species"].isin(EXPECTED_SPECIES)
)

# Stop rather than silently discard records with unknown target labels.
unknown_species_rows = clean_df.loc[unknown_species_mask].copy()
if not unknown_species_rows.empty:
    display(unknown_species_rows)
    raise ValueError(
        "Unknown species labels were found. Investigate and resolve them "
        "before creating the cleaned dataset."
    )

# Remove only rows with missing required values or invalid physical lengths.
rows_missing_removed = int(missing_required_mask.sum())
rows_invalid_removed = int((~missing_required_mask & invalid_measurement_mask).sum())

clean_df = clean_df.loc[
    ~(missing_required_mask | invalid_measurement_mask)
].copy()

# Keep duplicate observations unless their accidental duplication is confirmed.
clean_df = clean_df[expected_columns].reset_index(drop=True)

rows_after_cleaning = len(clean_df)

cleaning_summary = pd.DataFrame({
    "measure": [
        "Rows before cleaning",
        "Rows removed for missing required values",
        "Rows removed for non-positive measurements",
        "Rows after cleaning",
        "Rows retained"
    ],
    "count": [
        rows_before_cleaning,
        rows_missing_removed,
        rows_invalid_removed,
        rows_after_cleaning,
        rows_after_cleaning
    ]
})

display(cleaning_summary)

,measure,count
0,Rows before cleaning,150
1,Rows removed for missing required values,0
2,Rows removed for non-positive measurements,0
3,Rows after cleaning,150
4,Rows retained,150


# 16. Final Data-Quality Validation

After cleaning, repeat the key checks. A cleaned dataset should satisfy the expected schema and data rules.

Validation criteria:
- exactly the five required columns are present;
- measurement fields are numeric;
- no missing required values remain;
- all measurements are positive;
- species labels belong to the expected set;
- no unexpected rows or columns have appeared.

In [16]:
final_validation = {
    "expected_columns_match": clean_df.columns.tolist() == expected_columns,
    "all_measurements_numeric": all(
        pd.api.types.is_numeric_dtype(clean_df[col])
        for col in measurement_columns
    ),
    "missing_cells": int(clean_df.isna().sum().sum()),
    "non_positive_measurements": int(
        (clean_df[measurement_columns] <= 0).sum().sum()
    ),
    "unknown_species_labels": int(
        (~clean_df["species"].isin(EXPECTED_SPECIES)).sum()
    ),
    "duplicate_rows_retained_for_review": int(clean_df.duplicated().sum()),
    "final_rows": len(clean_df),
    "final_columns": clean_df.shape[1]
}

validation_report = pd.Series(final_validation, name="result").to_frame()
display(validation_report)

assert final_validation["expected_columns_match"], "Column schema mismatch."
assert final_validation["all_measurements_numeric"], "Non-numeric measurement column."
assert final_validation["missing_cells"] == 0, "Missing values remain."
assert final_validation["non_positive_measurements"] == 0, "Invalid measurements remain."
assert final_validation["unknown_species_labels"] == 0, "Unknown species labels remain."

print("All mandatory final validation checks passed.")

,result
expected_columns_match,True
all_measurements_numeric,True
missing_cells,0
non_positive_measurements,0
unknown_species_labels,0
duplicate_rows_retained_for_review,3
final_rows,150
final_columns,5


All mandatory final validation checks passed.


# 17. Before-and-After Comparison

This section summarises whether cleaning changed the dataset. The raw file remains untouched; only the processed DataFrame is exported.

A good cleaning workflow should be able to explain every row-count change. If no issues are present, it is valid for the before-and-after row counts to be identical.

In [17]:
before_after = pd.DataFrame({
    "stage": ["Raw loaded data", "Cleaned data"],
    "rows": [rows_before_cleaning, rows_after_cleaning],
    "columns": [len(expected_columns), clean_df.shape[1]]
})

before_after["rows_removed_from_raw"] = [
    0,
    rows_before_cleaning - rows_after_cleaning
]

display(before_after)
print("Row retention rate: "
      f"{(rows_after_cleaning / rows_before_cleaning * 100):.2f}%")

,stage,rows,columns,rows_removed_from_raw
0,Raw loaded data,150,5,0
1,Cleaned data,150,5,0


Row retention rate: 100.00%


# 18. Save the Cleaned Dataset and Quality Reports

The original file stays in `data/raw/iris.data`.

This section exports:
- `iris_cleaned.csv` — cleaned observations for the next notebook;
- `data_quality_report.csv` — key validation results;
- `species_summary.csv` — group-wise descriptive summary.

These are data outputs, not additional Jupyter notebooks.

In [18]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(CLEAN_FILE, index=False)

quality_report_path = PROCESSED_DIR / "data_quality_report.csv"
species_summary_path = PROCESSED_DIR / "species_summary.csv"

validation_report.to_csv(quality_report_path, header=["result"])
species_business_summary.to_csv(species_summary_path, index=False)

print("Cleaned dataset saved:", CLEAN_FILE.resolve())
print("Quality report saved:", quality_report_path.resolve())
print("Species summary saved:", species_summary_path.resolve())

Cleaned dataset saved: D:\Datheon-ML-Data-Science-Internship\Task-01-Iris-Classification\data\processed\iris_cleaned.csv
Quality report saved: D:\Datheon-ML-Data-Science-Internship\Task-01-Iris-Classification\data\processed\data_quality_report.csv
Species summary saved: D:\Datheon-ML-Data-Science-Internship\Task-01-Iris-Classification\data\processed\species_summary.csv


# 19. Reload the Exported File and Verify It

A successful `to_csv()` call does not by itself prove that the exported file can be read correctly. Reload it and compare its dimensions and column names with the in-memory cleaned DataFrame.

In [19]:
reloaded_clean_df = pd.read_csv(CLEAN_FILE)

print("Reloaded file shape:", reloaded_clean_df.shape)
print("In-memory cleaned shape:", clean_df.shape)
print("Columns match:", reloaded_clean_df.columns.tolist() == clean_df.columns.tolist())

assert reloaded_clean_df.shape == clean_df.shape
assert reloaded_clean_df.columns.tolist() == clean_df.columns.tolist()

display(reloaded_clean_df.head())
print("Export verification passed.")

Reloaded file shape: (150, 5)
In-memory cleaned shape: (150, 5)
Columns match: True


,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
0,5.100,3.500,1.400,0.200,Iris-setosa
1,4.900,3.000,1.400,0.200,Iris-setosa
2,4.700,3.200,1.300,0.200,Iris-setosa
3,4.600,3.100,1.500,0.200,Iris-setosa
4,5.000,3.600,1.400,0.200,Iris-setosa


Export verification passed.


# 20. Business Interpretation and Findings

Use the actual outputs generated above to write a short evidence-based summary.

### Suggested interpretation prompts

1. How many records and features were available before cleaning?
2. Were any required values missing?
3. Were any non-positive measurements detected?
4. Were species labels consistent with the expected three classes?
5. Were exact duplicate rows found, and why were they retained or removed?
6. Did cleaning change the row count?
7. Which species had the highest average petal length?
8. What does the group-wise summary suggest about differences between species?

### Business meaning

A nursery or research team could use the species-level summaries to understand how recorded measurements vary between species. However, group averages alone do not establish a reliable identification rule. Classification performance must be measured in the later modelling stage.

In [20]:
# Automatically prepare a concise findings table from the actual cleaned data.
findings = {
    "records_after_cleaning": len(clean_df),
    "species_count": clean_df["species"].nunique(),
    "missing_cells_after_cleaning": int(clean_df.isna().sum().sum()),
    "duplicate_rows_retained": int(clean_df.duplicated().sum()),
    "species_with_highest_average_petal_length": (
        clean_df.groupby("species")["petal_length_cm"].mean().idxmax()
    ),
    "highest_average_petal_length_cm": (
        clean_df.groupby("species")["petal_length_cm"].mean().max()
    )
}

display(pd.Series(findings, name="finding").to_frame())

,finding
records_after_cleaning,150
species_count,3
missing_cells_after_cleaning,0
duplicate_rows_retained,3
species_with_highest_average_petal_length,Iris-virginica
highest_average_petal_length_cm,5.552


# 21. Limitations and Good-Practice Notes

- Iris is a small, classic educational dataset; it is not representative of every real-world flower population.
- Duplicate measurements may be legitimate separate observations.
- IQR flags are screening indicators, not proof of errors.
- No synthetic rows or duplicated rows should be added to increase dataset size.
- Do not fit scalers or other learned preprocessing transformations on the full dataset before splitting.
- Keep the raw file unchanged so that every transformation can be reproduced.
- Record any manual correction with the reason and source of evidence.

# 22. Notebook Completion Checklist

- [ ] Loaded the original raw Iris file from `data/raw/`.
- [ ] Confirmed the expected five-column schema.
- [ ] Inspected data types and missing values.
- [ ] Investigated duplicate records.
- [ ] Validated species labels.
- [ ] Checked non-positive measurements.
- [ ] Reviewed descriptive statistics.
- [ ] Used `groupby()` and `agg()` for species-level summaries.
- [ ] Screened potential outliers without deleting them blindly.
- [ ] Applied only justified cleaning rules.
- [ ] Revalidated the cleaned dataset.
- [ ] Saved `iris_cleaned.csv` to `data/processed/`.
- [ ] Saved the quality and species summary reports.
- [ ] Reloaded the exported CSV and verified it.

# 23. Conclusion

This notebook establishes a reproducible data-quality workflow for the Iris classification project. It preserves the original raw dataset, inspects potential issues, applies conservative cleaning rules, validates the result, and exports a processed dataset for downstream analysis.

**Next notebook:** `03_Exploratory_Data_Analysis.ipynb`

Use `data/processed/iris_cleaned.csv` as the input for EDA. Do not repeat cleaning transformations unnecessarily; refer back to this notebook for the documented decisions.

# 24. References

1. UCI Machine Learning Repository — Iris Dataset: https://archive.ics.uci.edu/dataset/53/iris
2. Scikit-learn — Iris dataset description: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html
3. Pandas — Working with missing data: https://pandas.pydata.org/docs/user_guide/missing_data.html
4. Pandas — GroupBy user guide: https://pandas.pydata.org/docs/user_guide/groupby.html
5. Pandas — DataFrame aggregation: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html
6. Pandas — `read_csv()`: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
7. Pandas — `DataFrame.to_csv()`: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html

---
**Datheon Internship · Task 01 · Iris Classification**  
**Notebook 02:** Data Cleaning & Quality Assessment  
**Workflow boundary:** Inspect → Validate → Clean → Verify → Export